In [2]:
import sys
import os
import numpy as np

sys.path.append('..//')
from utils_mitgcm import open_mitgcm_ds_from_config
import pylake

/home/leroquan@eawag.wroot.emp-eaw.ch/miniconda3/envs/horizontal_structures/lib/python3.11/site-packages/pylake/pylake.py:3: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater as sw


In [3]:
model = 'geneva_2025'
mitgcm_config, ds = open_mitgcm_ds_from_config('..//config.json', model)

In [4]:
folder_path = os.path.dirname(mitgcm_config['datapath'])
output_folder = os.path.join(folder_path, "figures", "isotherm_map")
os.makedirs(output_folder, exist_ok=True)

In [5]:
grid_resolution = 100
ds['YC'] = np.arange(1, len(ds['YC']) + 1) * grid_resolution - grid_resolution / 2
ds['XC'] = np.arange(1, len(ds['XC']) + 1) * grid_resolution - grid_resolution / 2
ds['YG'] = np.arange(0, len(ds['YG'])) * grid_resolution
ds['XG'] = np.arange(0, len(ds['XG'])) * grid_resolution

# Spatial spectrum

In [6]:
t_idx = 5221
z_idx = 0

In [7]:
u = ds.UVEL.isel(Z=z_idx, time=t_idx)
v = ds.VVEL.isel(Z=z_idx, time=t_idx)

# remove mean
u = u - u.mean(dim=("XG","YC"))
v = v - v.mean(dim=("XC","YG"))

# FFT
u_hat = np.fft.fft2(u)
v_hat = np.fft.fft2(v)

# Energy
E2D = 0.5 * (np.abs(u_hat)**2 + np.abs(v_hat)**2)

In [8]:
nx = u.sizes["XG"]
ny = u.sizes["YC"]

kx = np.fft.fftfreq(nx, d=grid_resolution) * 2*np.pi
ky = np.fft.fftfreq(ny, d=grid_resolution) * 2*np.pi

In [9]:
kx_grid, ky_grid = np.meshgrid(kx, ky, indexing="ij")
k = np.sqrt(kx_grid**2 + ky_grid**2)

k_bins = np.linspace(0, k.max(), 50)
E_k = np.zeros(len(k_bins)-1)

for i in range(len(k_bins)-1):
    mask = (k >= k_bins[i]) & (k < k_bins[i+1])
    E_k[i] = E2D[mask].mean()

IndexError: boolean index did not match indexed array along axis 0; size of axis is 245 but size of corresponding boolean axis is 660